# 00_setup - Aprovisionamiento parametrizado FinPay Lakehouse

Este notebook prepara el ambiente según el catálogo indicado por widgets.

Parámetros:
- `env`: `dev` o `prod`.
- `catalog`: catálogo objetivo. Usar `fintech_finpay_dev` para DEV y `fintech_finpay` para PROD.
- `landing_volume`: volumen de landing. Por defecto: `vol_landing`.
- `group_ingenieria`: grupo con permisos de ingeniería.
- `group_riesgo`: grupo con permisos de lectura sobre Silver/Gold.
- `group_auditoria`: grupo con permisos de lectura sobre Gold/Observability.

Criterios:
- Bronze: columnas de negocio en `STRING`.
- Silver: columnas tipadas según semántica.
- `ingestion_archetypes.json` generado dinámicamente por ambiente.
- Las reglas de column masking y row-level security se aplican después con `00_apply_security_rules.ipynb`.


In [ ]:
# COMMAND ----------
# DBTITLE 1,Parámetros del ambiente

dbutils.widgets.text("env", "dev")
dbutils.widgets.text("catalog", "fintech_finpay_dev")
dbutils.widgets.text("landing_volume", "vol_landing")

dbutils.widgets.text("group_ingenieria", "finpay_ingenieria")
dbutils.widgets.text("group_riesgo", "finpay_riesgo")
dbutils.widgets.text("group_auditoria", "finpay_auditoria")

ENV = dbutils.widgets.get("env").strip().lower()
CATALOG = dbutils.widgets.get("catalog").strip()

SCHEMA_DEFAULT = "default"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"
SCHEMA_OBSERVABILITY = "observability"

VOLUME_NAME = dbutils.widgets.get("landing_volume").strip()
LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA_DEFAULT}/{VOLUME_NAME}"

GROUP_INGENIERIA = dbutils.widgets.get("group_ingenieria").strip()
GROUP_RIESGO = dbutils.widgets.get("group_riesgo").strip()
GROUP_AUDITORIA = dbutils.widgets.get("group_auditoria").strip()

CURRENT_USER = spark.sql("SELECT current_user() AS user").first()["user"]

print(f"ENV                : {ENV}")
print(f"CATALOG            : {CATALOG}")
print(f"LANDING_PATH       : {LANDING_PATH}")
print(f"GROUP_INGENIERIA   : {GROUP_INGENIERIA}")
print(f"GROUP_RIESGO       : {GROUP_RIESGO}")
print(f"GROUP_AUDITORIA    : {GROUP_AUDITORIA}")
print(f"CURRENT_USER       : {CURRENT_USER}")


In [ ]:
# COMMAND ----------
# DBTITLE 1,Crear catálogo, schemas y volumen

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")

for schema_name in [SCHEMA_DEFAULT, SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD, SCHEMA_OBSERVABILITY]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name}")

spark.sql(f'''
CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME}
COMMENT 'Landing zone {ENV.upper()} para archivos fuente del proyecto FinPay'
''')

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))
display(spark.sql(f"SHOW VOLUMES IN {CATALOG}.{SCHEMA_DEFAULT}"))


In [ ]:
# COMMAND ----------
# DBTITLE 1,Crear carpetas de landing y metadata

directories = [
    f"{LANDING_PATH}/transactions",
    f"{LANDING_PATH}/merchants",
    f"{LANDING_PATH}/users",
    f"{LANDING_PATH}/metadata",
    f"{LANDING_PATH}/metadata/schema",
    f"{LANDING_PATH}/metadata/schema/transactions",
    f"{LANDING_PATH}/metadata/schema/merchants",
    f"{LANDING_PATH}/metadata/schema/users",
    f"{LANDING_PATH}/metadata/checkpoints",
    f"{LANDING_PATH}/metadata/checkpoints/transactions",
    f"{LANDING_PATH}/metadata/checkpoints/merchants",
    f"{LANDING_PATH}/metadata/checkpoints/users",
    f"{LANDING_PATH}/quarantine",
]

for path in directories:
    dbutils.fs.mkdirs(path)

display(dbutils.fs.ls(LANDING_PATH))


In [ ]:
# COMMAND ----------
# DBTITLE 1,Generar ingestion_archetypes.json parametrizado

import json

ingestion_archetypes = [
    {
        "source_name": "transactions",
        "source_path": f"{LANDING_PATH}/transactions/",
        "file_format": "csv",
        "delimiter": ",",
        "header": True,
        "multiline": False,
        "schema_location": f"{LANDING_PATH}/metadata/schema/transactions/",
        "checkpoint_path": f"{LANDING_PATH}/metadata/checkpoints/transactions/",
        "partition_by": "transaction_date",
        "target_table": f"{CATALOG}.{SCHEMA_BRONZE}.transactions",
        "active": True
    },
    {
        "source_name": "merchants",
        "source_path": f"{LANDING_PATH}/merchants/",
        "file_format": "json",
        "delimiter": None,
        "header": None,
        "multiline": True,
        "schema_location": f"{LANDING_PATH}/metadata/schema/merchants/",
        "checkpoint_path": f"{LANDING_PATH}/metadata/checkpoints/merchants/",
        "partition_by": "country",
        "target_table": f"{CATALOG}.{SCHEMA_BRONZE}.merchants",
        "active": True
    },
    {
        "source_name": "users",
        "source_path": f"{LANDING_PATH}/users/",
        "file_format": "csv",
        "delimiter": "|",
        "header": True,
        "multiline": False,
        "schema_location": f"{LANDING_PATH}/metadata/schema/users/",
        "checkpoint_path": f"{LANDING_PATH}/metadata/checkpoints/users/",
        "partition_by": "country",
        "target_table": f"{CATALOG}.{SCHEMA_BRONZE}.users",
        "active": True
    }
]

metadata_path = f"{LANDING_PATH}/metadata/ingestion_archetypes.json"

dbutils.fs.put(
    metadata_path,
    json.dumps(ingestion_archetypes, indent=2, ensure_ascii=False),
    overwrite=True
)

print(f"Archivo creado: {metadata_path}")
print(dbutils.fs.head(metadata_path, 5000))


In [ ]:
# COMMAND ----------
# DBTITLE 1,Asignar permisos por rol

def grant(sql_statement: str):
    print(sql_statement)
    spark.sql(sql_statement)

for principal in [GROUP_INGENIERIA, GROUP_RIESGO, GROUP_AUDITORIA]:
    grant(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{principal}`")

for schema_name in [SCHEMA_DEFAULT, SCHEMA_BRONZE, SCHEMA_SILVER, SCHEMA_GOLD, SCHEMA_OBSERVABILITY]:
    grant(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT CREATE TABLE ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT CREATE FUNCTION ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT MODIFY ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")
    grant(f"GRANT SELECT ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_INGENIERIA}`")

for schema_name in [SCHEMA_SILVER, SCHEMA_GOLD]:
    grant(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_RIESGO}`")
    grant(f"GRANT SELECT ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_RIESGO}`")

for schema_name in [SCHEMA_GOLD, SCHEMA_OBSERVABILITY]:
    grant(f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_AUDITORIA}`")
    grant(f"GRANT SELECT ON SCHEMA {CATALOG}.{schema_name} TO `{GROUP_AUDITORIA}`")

grant(f"GRANT READ VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_INGENIERIA}`")
grant(f"GRANT WRITE VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_INGENIERIA}`")

grant(f"GRANT READ VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_RIESGO}`")
grant(f"GRANT READ VOLUME ON VOLUME {CATALOG}.{SCHEMA_DEFAULT}.{VOLUME_NAME} TO `{GROUP_AUDITORIA}`")


In [ ]:
# COMMAND ----------
# DBTITLE 1,Validaciones finales

print("=== Schemas ===")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

print("=== Landing folders ===")
display(dbutils.fs.ls(LANDING_PATH))

print("=== ingestion_archetypes.json ===")
print(dbutils.fs.head(f"{LANDING_PATH}/metadata/ingestion_archetypes.json", 5000))

print("=== Bronze tables ===")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_BRONZE}"))

print("=== Silver tables ===")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_SILVER}"))

print("=== Observability tables ===")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_OBSERVABILITY}"))

print(f"Setup {ENV.upper()} finalizado correctamente para catálogo {CATALOG}.")
